# Milestone 1 — Dual-T4 TP performance diagnostics (self-contained Kaggle launcher)

This notebook recreates the Milestone 1 launcher without requiring a manually attached
`kaggle-vllm` source Dataset.

It deliberately uses the reviewed feature-branch commit:

`7e7355d64266e864a8113c30d52c612d98100350`

Execution flow:

1. validate the Kaggle Python / PyTorch / CUDA / dual-T4 baseline;
2. optionally load `HF_TOKEN` from Kaggle Secrets;
3. download the exact reviewed `kaggle-vllm` source archive from GitHub;
4. install only the lightweight SDK from that source into `/kaggle/working`;
5. resolve the existing immutable Qwen TP=2 `sharded_state` artifact from an attached
   Kaggle Input **or** download it from Hugging Face revision
   `08bb62d0b68d20062e9009a9769c0df53d3dae21`;
6. preview the benchmark matrix with a dry run;
7. execute the real offline-vLLM matrix on Kaggle T4×2;
8. preserve JSON/log/topology/checksum evidence and create a downloadable ZIP.

The native vLLM CUDA wheel is **not** rebuilt here. The Milestone runner invokes the
existing `kaggle-vllm bootstrap` flow, which resolves the immutable wheel from
`waqasm86/kaggle-vllm-binaries`, verifies its SHA256, stages it with `--no-deps`, and
preserves Kaggle's preinstalled PyTorch/CUDA stack.

This notebook does not fabricate results and does not claim that PCIe/NCCL alone causes
any measured TP=1/TP=2 delta.


In [2]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import urllib.request

SOURCE_IDENTITY = "7e7355d64266e864a8113c30d52c612d98100350"
SOURCE_ARCHIVE_URL = (
    "https://github.com/kaggle-vllm/kaggle-vllm/archive/"
    f"{SOURCE_IDENTITY}.tar.gz"
)

MODEL_REPO = "waqasm86/kaggle-vllm-models"
MODEL_REVISION = "08bb62d0b68d20062e9009a9769c0df53d3dae21"

EXPECTED_SDK_VERSION = "0.2.0"
EXPECTED_NATIVE_REPO = "waqasm86/kaggle-vllm-binaries"
EXPECTED_NATIVE_REVISION = "f6b4f10de54924ed6fe9e28cceab84eca7276ab6"
EXPECTED_NATIVE_WHEEL = (
    "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-"
    "cp312-cp312-linux_x86_64.whl"
)
EXPECTED_NATIVE_SHA256 = (
    "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"
)

WORK = Path("/kaggle/working/kaggle-vllm-milestone-1-self-contained")
SOURCE_DOWNLOAD = WORK / "source.tar.gz"
SOURCE_EXTRACT = WORK / "source"
SDK_TARGET = WORK / "sdk"
QWEN_DOWNLOAD = WORK / "qwen2.5-3b-t4x2-sharded"
EVIDENCE_DIR = Path("/kaggle/working/kaggle-vllm-tp-milestone-1")
ARCHIVE_BASE = Path("/kaggle/working/kaggle-vllm-tp-milestone-1-evidence")

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Source identity:", SOURCE_IDENTITY)
print("Evidence dir:", EVIDENCE_DIR)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Source identity: 7e7355d64266e864a8113c30d52c612d98100350
Evidence dir: /kaggle/working/kaggle-vllm-tp-milestone-1


## 1. Validate the Kaggle dual-T4 baseline

Use a fresh Kaggle session with the **T4 ×2** accelerator selected.

The exact compatibility profile used by the current native artifact is intentionally
narrow. This preflight fails early if Kaggle has changed the key Python/PyTorch/CUDA/GPU
contract.


In [3]:
import torch

subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i),
        "SM",
        torch.cuda.get_device_capability(i),
        "VRAM GiB",
        round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2),
    )

assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__ == "2.10.0+cu128", torch.__version__
assert torch.version.cuda == "12.8", torch.version.cuda
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2
assert all("Tesla T4" in torch.cuda.get_device_name(i) for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

TORCH_BEFORE = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
print("Dual-T4 baseline: PASS")


GPU 0: Tesla T4 (UUID: GPU-f9ee88c5-fd63-6f19-bfa9-bb88159b8fc0)
GPU 1: Tesla T4 (UUID: GPU-07931a3a-7b28-c686-2bbc-e26d4b3e97a9)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
Torch: 2.10.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 SM (7, 5) VRAM GiB 14.56
GPU 1: Tesla T4 SM (7, 5) VRAM GiB 14.56
Dual-T4 baseline: PASS


## 2. Optional Hugging Face authentication

Both project repositories may be publicly readable, so a token is not mandatory when
public access is available. If you have stored `HF_TOKEN` in Kaggle Secrets, this cell
uses it automatically.


In [5]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN loaded from Kaggle Secrets: YES")
    else:
        print("HF_TOKEN loaded: NO; continuing with public access")
except Exception as exc:
    print("HF_TOKEN unavailable; continuing with public access.")
    print("Reason:", type(exc).__name__)


HF_TOKEN unavailable; continuing with public access.
Reason: BackendError


## 3. Download the exact reviewed `kaggle-vllm` source from GitHub

This replaces the old notebook's failing assumption that a manually created source
Dataset exists under `/kaggle/input`.

Only notebook-owned paths under `/kaggle/working/kaggle-vllm-milestone-1-self-contained`
are reset here.


In [6]:
if WORK.exists():
    print("Removing previous notebook-owned work directory:", WORK)
    shutil.rmtree(WORK)

WORK.mkdir(parents=True)
SOURCE_EXTRACT.mkdir(parents=True)

print("Downloading:", SOURCE_ARCHIVE_URL)
urllib.request.urlretrieve(SOURCE_ARCHIVE_URL, SOURCE_DOWNLOAD)

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_ARCHIVE_SHA256 = sha256_file(SOURCE_DOWNLOAD)
print("Source archive SHA256:", SOURCE_ARCHIVE_SHA256)

with tarfile.open(SOURCE_DOWNLOAD, "r:gz") as archive:
    archive.extractall(SOURCE_EXTRACT)

roots = [
    p for p in SOURCE_EXTRACT.iterdir()
    if p.is_dir() and (p / "pyproject.toml").is_file()
]
assert len(roots) == 1, roots
SOURCE_ROOT = roots[0].resolve()

assert (SOURCE_ROOT / "pyproject.toml").is_file()
assert (SOURCE_ROOT / "scripts" / "kaggle_tp_diagnostics.py").is_file()
assert (SOURCE_ROOT / "src" / "kaggle_vllm").is_dir()

print("Resolved source root:", SOURCE_ROOT)
print("Reviewed source identity:", SOURCE_IDENTITY)


Downloading: https://github.com/kaggle-vllm/kaggle-vllm/archive/7e7355d64266e864a8113c30d52c612d98100350.tar.gz
Source archive SHA256: df79fc65fe66989fb91e781eb99c5ff286ef1dea2daaf441999aee9115feefe5
Resolved source root: /kaggle/working/kaggle-vllm-milestone-1-self-contained/source/kaggle-vllm-7e7355d64266e864a8113c30d52c612d98100350
Reviewed source identity: 7e7355d64266e864a8113c30d52c612d98100350


/tmp/ipykernel_58/973428337.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(SOURCE_EXTRACT)


## 4. Install the lightweight SDK from the reviewed source

This installs the SDK and its small `hub` helper dependency into a notebook-owned target.
It does **not** install vLLM, Torch, CUDA, or the model as normal package dependencies.

The actual native vLLM CUDA runtime is staged later by `kaggle-vllm bootstrap`.


In [7]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--target",
        str(SDK_TARGET),
        f"{SOURCE_ROOT}[hub]",
    ],
    check=True,
)

RUN_ENV = dict(os.environ)
RUN_ENV["PYTHONPATH"] = (
    f"{SDK_TARGET}{os.pathsep}{RUN_ENV['PYTHONPATH']}"
    if RUN_ENV.get("PYTHONPATH")
    else str(SDK_TARGET)
)

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import kaggle_vllm; "
            "print(kaggle_vllm.__version__); "
            "print(kaggle_vllm.__file__)"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
)
print(probe.stdout)
assert probe.stdout.splitlines()[0].strip() == EXPECTED_SDK_VERSION

RUNNER = SOURCE_ROOT / "scripts" / "kaggle_tp_diagnostics.py"
assert RUNNER.is_file()
print("Runner:", RUNNER)


Processing ./kaggle-vllm-milestone-1-self-contained/source/kaggle-vllm-7e7355d64266e864a8113c30d52c612d98100350
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 315.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 314.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.0/100.0 kB 326.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 343.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 191.7 MB/s eta 0:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.7.0 which is incompatible.


0.2.0
/kaggle/working/kaggle-vllm-milestone-1-self-contained/sdk/kaggle_vllm/__init__.py

Runner: /kaggle/working/kaggle-vllm-milestone-1-self-contained/source/kaggle-vllm-7e7355d64266e864a8113c30d52c612d98100350/scripts/kaggle_tp_diagnostics.py


## 5. Resolve the existing immutable Qwen TP=2 artifact

Resolution order:

1. use `KAGGLE_VLLM_QWEN_PATH` if you explicitly set it;
2. search attached `/kaggle/input` Datasets for the validated four-rank/part layout;
3. reuse an already downloaded copy in this notebook-owned work directory;
4. otherwise download the existing published Hugging Face snapshot at the exact immutable
   revision.

This downloads the already-created vLLM `sharded_state`; it does **not** regenerate or
fine-tune Qwen.


In [8]:
from huggingface_hub import snapshot_download

EXPECTED_QWEN_SHARDS = {
    "model-rank-0-part-0.safetensors": 2_138_586_872,
    "model-rank-0-part-1.safetensors": 947_544_384,
    "model-rank-1-part-0.safetensors": 2_138_586_872,
    "model-rank-1-part-1.safetensors": 947_544_384,
}

def valid_qwen_dir(candidate: Path) -> bool:
    try:
        candidate = candidate.resolve()
        if not candidate.is_dir():
            return False
        names = {p.name for p in candidate.iterdir()}
        if not set(EXPECTED_QWEN_SHARDS).issubset(names):
            return False
        if not (candidate / "config.json").is_file():
            return False
        if not (candidate / "tokenizer_config.json").is_file():
            return False
        for name, expected_size in EXPECTED_QWEN_SHARDS.items():
            p = candidate / name
            if not p.is_file() or p.stat().st_size != expected_size:
                return False
        return True
    except OSError:
        return False

qwen_path = None
resolution_method = None

explicit = os.environ.get("KAGGLE_VLLM_QWEN_PATH")
if explicit:
    candidate = Path(explicit)
    if valid_qwen_dir(candidate):
        qwen_path = candidate.resolve()
        resolution_method = "KAGGLE_VLLM_QWEN_PATH"

if qwen_path is None:
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for shard0 in sorted(input_root.rglob("model-rank-0-part-0.safetensors")):
            if valid_qwen_dir(shard0.parent):
                qwen_path = shard0.parent.resolve()
                resolution_method = "attached-kaggle-input"
                break

if qwen_path is None and valid_qwen_dir(QWEN_DOWNLOAD):
    qwen_path = QWEN_DOWNLOAD.resolve()
    resolution_method = "existing-working-copy"

if qwen_path is None:
    print("No valid attached Qwen TP=2 artifact found.")
    print("Downloading immutable Hugging Face snapshot:", MODEL_REPO, MODEL_REVISION)
    downloaded = snapshot_download(
        repo_id=MODEL_REPO,
        revision=MODEL_REVISION,
        local_dir=str(QWEN_DOWNLOAD),
        token=HF_TOKEN,
    )
    qwen_path = Path(downloaded).resolve()
    resolution_method = "huggingface-snapshot-download"

assert valid_qwen_dir(qwen_path), qwen_path

print("Qwen resolution method:", resolution_method)
print("Qwen path:", qwen_path)
for name in sorted(EXPECTED_QWEN_SHARDS):
    p = qwen_path / name
    print(name, p.stat().st_size, "symlink:", p.is_symlink())


No valid attached Qwen TP=2 artifact found.


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Qwen resolution method: huggingface-snapshot-download
Qwen path: /kaggle/working/kaggle-vllm-milestone-1-self-contained/qwen2.5-3b-t4x2-sharded
model-rank-0-part-0.safetensors 2138586872 symlink: False
model-rank-0-part-1.safetensors 947544384 symlink: False
model-rank-1-part-0.safetensors 2138586872 symlink: False
model-rank-1-part-1.safetensors 947544384 symlink: False


## 6. Audit the Milestone 1 plan

The dry run does not bootstrap the CUDA runtime, instantiate vLLM engines, or create the
evidence directory.

It should show four OPT-125M control rows and two Qwen TP=2 rows.


In [9]:
# Use a fresh evidence directory for an executed run.
if EVIDENCE_DIR.exists():
    raise RuntimeError(
        f"{EVIDENCE_DIR} already exists. "
        "Use a fresh Kaggle session, download the prior evidence, or explicitly "
        "choose a new EVIDENCE_DIR before rerunning."
    )

base_command = [
    sys.executable,
    str(RUNNER),
    "--source-identity",
    SOURCE_IDENTITY,
    "--expected-sdk-version",
    EXPECTED_SDK_VERSION,
    "--qwen-model",
    str(qwen_path),
    "--output-dir",
    str(EVIDENCE_DIR),
]

dry = subprocess.run(
    [*base_command, "--dry-run"],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
)
print(dry.stdout)

plan = json.loads(dry.stdout)
run_names = [row["name"] for row in plan["runs"]]
print("Planned rows:", run_names)

expected_rows = {
    "opt125m-tp1-eager0",
    "opt125m-tp2-eager0",
    "opt125m-tp1-eager1",
    "opt125m-tp2-eager1",
    "qwen-tp2-baseline",
    "qwen-tp2-batched-4096",
}
assert set(run_names) == expected_rows, run_names
assert plan["mutations_performed"] is False
print("Milestone 1 dry-run plan: PASS")


{
  "status": "planned_not_executed",
  "output_directory": "/kaggle/working/kaggle-vllm-tp-milestone-1",
  "expected_sdk_version": "0.2.0",
  "qwen_model_configured": true,
  "runs": [
    {
      "name": "opt125m-tp1-eager0",
      "plan": {
        "schema_version": 1,
        "status": "planned_not_executed",
        "measurement_mode": "offline_llm_generate",
        "engine": {
          "model": "facebook/opt-125m",
          "tensor_parallel_size": 1,
          "model_revision": "27dcfa74d334bc871f3234de431e71c6eeba5dd6",
          "model_representation": "transformers",
          "load_format": null,
          "dtype": "float16",
          "max_model_len": 512,
          "gpu_memory_utilization": 0.4,
          "enforce_eager": false,
          "disable_custom_all_reduce": true,
          "max_num_batched_tokens": null,
          "max_num_seqs": null,
          "workload": {
            "prompts": [
              "Explain tensor parallel inference in one paragraph.",
         

## 7. Execute the real Kaggle T4×2 matrix

This is the GPU-expensive step.

The runner:

- performs strict compatibility validation;
- downloads and SHA256-verifies the immutable native vLLM CUDA wheel from
  `waqasm86/kaggle-vllm-binaries`;
- stages it without replacing Kaggle's PyTorch/CUDA;
- captures NVIDIA topology;
- runs each benchmark row in an isolated child process;
- writes small machine-readable evidence under `/kaggle/working`.

If a row fails, preserve the failure JSON/log rather than deleting unfavorable evidence.


In [10]:
result = subprocess.run(
    base_command,
    check=False,
    text=True,
    env=RUN_ENV,
)

print("Milestone runner return code:", result.returncode)

if result.returncode != 0:
    print("\nRunner failed. Preserve the evidence directory and inspect logs:")
    if EVIDENCE_DIR.exists():
        for p in sorted(EVIDENCE_DIR.iterdir()):
            print(" ", p.name, p.stat().st_size if p.is_file() else "<dir>")
    raise RuntimeError(
        f"Milestone 1 runner failed with return code {result.returncode}"
    )

assert (EVIDENCE_DIR / "summary.json").is_file()
assert (EVIDENCE_DIR / "SHA256SUMS.txt").is_file()

print("\n=== summary.json ===")
print((EVIDENCE_DIR / "summary.json").read_text(encoding="utf-8"))

print("\n=== SHA256SUMS.txt ===")
print((EVIDENCE_DIR / "SHA256SUMS.txt").read_text(encoding="utf-8"))


Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 155.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 313.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 331.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 284.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 256.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 247.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 303.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 330.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 8. Verify runtime provenance and preserve the evidence bundle

This final cell records the notebook-level source/Qwen/native identities, checks that the
system Torch installation was preserved, and creates a ZIP for manual download from
Kaggle Output.

The ZIP is not uploaded anywhere automatically.


In [11]:
TORCH_AFTER = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
print("Torch before:", TORCH_BEFORE)
print("Torch after :", TORCH_AFTER)
assert TORCH_AFTER == TORCH_BEFORE

run_metadata = json.loads(
    (EVIDENCE_DIR / "run-metadata.json").read_text(encoding="utf-8")
)

assert run_metadata["milestone_source_identity"] == SOURCE_IDENTITY
native = run_metadata["native_runtime"]
assert native["wheel"] == EXPECTED_NATIVE_WHEEL
assert native["sha256"] == EXPECTED_NATIVE_SHA256
assert native["hf_repository"] == EXPECTED_NATIVE_REPO
assert native["hf_revision"] == EXPECTED_NATIVE_REVISION

notebook_provenance = {
    "status": "PASS",
    "source_identity": SOURCE_IDENTITY,
    "source_archive_url": SOURCE_ARCHIVE_URL,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "sdk_version": EXPECTED_SDK_VERSION,
    "native_repository": EXPECTED_NATIVE_REPO,
    "native_revision": EXPECTED_NATIVE_REVISION,
    "native_wheel": EXPECTED_NATIVE_WHEEL,
    "native_sha256": EXPECTED_NATIVE_SHA256,
    "qwen_repository": MODEL_REPO,
    "qwen_revision": MODEL_REVISION,
    "qwen_resolution_method": resolution_method,
    "qwen_path": str(qwen_path),
    "torch_before": TORCH_BEFORE,
    "torch_after": TORCH_AFTER,
}
(EVIDENCE_DIR / "notebook-provenance.json").write_text(
    json.dumps(notebook_provenance, indent=2) + "\n",
    encoding="utf-8",
)

# Rebuild checksums to include notebook-provenance.json.
checksum_file = EVIDENCE_DIR / "SHA256SUMS.txt"
checksum_file.unlink(missing_ok=True)
files = sorted(
    p for p in EVIDENCE_DIR.iterdir()
    if p.is_file() and p.name != "SHA256SUMS.txt"
)
checksum_file.write_text(
    "".join(f"{sha256_file(p)}  {p.name}\n" for p in files),
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(ARCHIVE_BASE),
    "zip",
    root_dir=EVIDENCE_DIR,
)

print(json.dumps(notebook_provenance, indent=2))
print("\nEvidence directory:", EVIDENCE_DIR)
print("Evidence archive:", archive_path)
print("Evidence archive SHA256:", sha256_file(Path(archive_path)))
print("\nFINAL MILESTONE 1 NOTEBOOK: PASS")


Torch before: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
Torch after : {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
{
  "status": "PASS",
  "source_identity": "7e7355d64266e864a8113c30d52c612d98100350",
  "source_archive_url": "https://github.com/kaggle-vllm/kaggle-vllm/archive/7e7355d64266e864a8113c30d52c612d98100350.tar.gz",
  "source_archive_sha256": "df79fc65fe66989fb91e781eb99c5ff286ef1dea2daaf441999aee9115feefe5",
  "sdk_version": "0.2.0",
  "native_repository": "waqasm86/kaggle-vllm-binaries",
  "native_revision": "f6b4f10de54924ed6fe9e28cceab84eca7276ab6",
  "native_wheel": "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl",
  "native_sha256": "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c",
  "qwen_repository": "waqasm86/kaggle-vllm-models",
  "qwen_revision": "08bb62d0b68d20062e9009a9769c0df53d3dae